# Road-Only Model (Model A)

Logistic regression on the 6 road features from notebook 05. We compare it to a simple baseline on
an area the model never saw during training.

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.impute import MissingIndicator, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cwd = Path.cwd()
project_root = next(p for p in [cwd, *cwd.parents] if (p / ".git").exists())
PROCESSED = project_root / "data" / "processed"

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#52514e"

df = pd.read_csv(PROCESSED / "model_features.csv")

## 1. Segments

We drop the 393 "Unknown" road type segments. None of them had a crash, so the model would just learn
"Unknown = safe," which isn't real.

In [ ]:
df = df[df["road_type"] != "Unknown"].reset_index(drop=True)
print(f"Segments: {len(df):,}")
print(f"With a crash: {df['had_crash'].sum():,} ({df['had_crash'].mean():.1%})")

## 2. Train/test split by area

Nearby streets are alike, so a random split would let the model see the test area's neighbors during
training. Instead we cut Boston into 2 km × 2 km squares and put a random 25% of the squares in the
test set. Each segment goes in the square its center falls in.

In [ ]:
road_runs = sorted(PROCESSED.glob("boston_cleaning_*/retained_source_features_esri.json"))
shapes = json.loads(road_runs[-1].read_text())

centers = {}
for f in shapes["features"]:
    points = np.array([pt[:2] for path in f["geometry"]["paths"] for pt in path])
    centers[f["attributes"]["GlobalID"]] = points.mean(axis=0)

xy = np.array([centers[s] for s in df["segment_id"]])
df["x"], df["y"] = xy[:, 0], xy[:, 1]

BLOCK = 2000  # meters
df["block"] = (df["x"] // BLOCK).astype(int).astype(str) + "_" + (df["y"] // BLOCK).astype(int).astype(str)

blocks = np.sort(df["block"].unique())
rng = np.random.default_rng(506)
test_blocks = rng.choice(blocks, size=round(len(blocks) * 0.25), replace=False)
df["split"] = np.where(df["block"].isin(test_blocks), "test", "train")

df.groupby("split")["had_crash"].agg(segments="size", crashes="sum", crash_rate="mean").round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for name, color in [("train", "#c3c2b7"), ("test", BLUE)]:
    part = df[df["split"] == name]
    ax.scatter(part["x"], part["y"], s=1, color=color, label=name)
ax.set_title("Train and test areas", loc="left")
ax.legend(frameon=False, markerscale=8)
ax.set_aspect("equal")
ax.axis("off")
plt.show()

The test area's crash rate is a bit higher than training (10% vs. 8%), so PR-AUC numbers here will
look a little better than they would on an average area. Both models are scored on the same test set,
so the comparison is still fair.

In [ ]:
train, test = df[df["split"] == "train"], df[df["split"] == "test"]

## 3. Baseline

No model. Each test segment's guess is how often its road type crashed in the training area.

In [ ]:
road_type_rate = train.groupby("road_type")["had_crash"].mean()
baseline_guess = test["road_type"].map(road_type_rate)
road_type_rate.mul(100).round(1).sort_values(ascending=False).rename("crash rate %")

## 4. Inputs

Logistic regression only takes numbers, no blanks, and works best when the inputs are on a similar scale.

| Feature | What we do to it |
|---|---|
| exposure | log, since it goes from 0 to 298,000 and a few highways would take over |
| exposure, width | fill blanks with the training median, add a "was missing" 0/1 column |
| exposure, width, connections | rescale so they're on the same scale |
| road type, sidewalks | turn into 0/1 columns. Local and "both sidewalks" are the default, so they get no column |
| connections × arterial | connections if the road is an arterial, 0 if not. Lets connections go up on local streets and down on arterials (notebook 05, section 7) |
| traffic counted | already 0/1 |

The medians and scaling come from the training area only, so the test area doesn't leak in. The
pipeline below handles that.

In [ ]:
df["log_exposure"] = np.log1p(df["exposure"])
df["connections_x_arterial"] = df["connections"] * df["road_type"].str.contains("arterial")
train, test = df[df["split"] == "train"], df[df["split"] == "test"]

inputs = ColumnTransformer([
    ("fill", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), ["log_exposure", "surface_wd_clean"]),
    ("missing", MissingIndicator(features="all"), ["log_exposure", "surface_wd_clean"]),
    ("scale", StandardScaler(), ["connections", "connections_x_arterial"]),
    ("keep", "passthrough", ["traffic_counted"]),
    ("onehot", OneHotEncoder(drop=["Local", "both"]), ["road_type", "sidewalks"]),
], verbose_feature_names_out=False)

model = make_pipeline(inputs, LogisticRegression(max_iter=1000))

Here's what 5 random training rows look like after this step. Scaled columns are centered on 0, so
0 means average and 1 means one standard deviation above average.

In [ ]:
inputs.fit(train)
pd.DataFrame(inputs.transform(train.sample(5, random_state=1)), columns=inputs.get_feature_names_out()).round(2)

## 5. Train

In [ ]:
model.fit(train, train["had_crash"])
model_guess = model.predict_proba(test)[:, 1]

## 6. Results

Why PR-AUC:
- Only about 9% of segments had a crash. A model that always says "no crash" would be 91% accurate and
  useless, so accuracy doesn't work here.
- PR-AUC only cares about finding the crash segments: of the roads we flag, how many crashed
  (precision), and of the roads that crashed, how many we flagged (recall).
- It checks every cutoff, so we don't have to pick one yet.
- It's the measure the proposal uses to compare Model A and Model B.

It goes from 0 to 1. Guessing at random scores about the test crash rate (0.10), so that's the floor.

In [ ]:
results = pd.DataFrame({
    "model": ["random guess", "baseline (road type rate)", "logistic regression"],
    "PR-AUC": [test["had_crash"].mean(),
               average_precision_score(test["had_crash"], baseline_guess),
               average_precision_score(test["had_crash"], model_guess)],
}).round(3)
results

Logistic regression roughly doubles the baseline (0.35 vs. 0.19). So the other features add a lot
beyond road type.

## 7. What the model learned

Each input's weight. Positive pushes the crash chance up, negative pushes it down. Road types are
compared to local streets.

In [ ]:
weights = pd.Series(model[-1].coef_[0], index=model[0].get_feature_names_out()).sort_values()

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(weights.index, weights, color=[BLUE if w > 0 else ORANGE for w in weights], height=0.6)
ax.axvline(0, color=GRAY, lw=1)
ax.set_title("Model weights", loc="left")
ax.set_xlabel("← lower crash chance     higher crash chance →")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

- Every road type is riskier than local streets, with principal arterials the highest. Interstates
  aren't the top anymore, because most of their risk is explained by their traffic.
- Exposure is the strongest number input: more traffic and length, more crashes.
- Traffic counted is strongly negative, even though counted roads crashed more in notebook 05. That's
  not saying counted roads are safer. Roads without a count use the 1,154 default AADT, which probably
  undercounts their real traffic. So at the same exposure, an uncounted road crashes about as often or
  more. This weight is the model correcting their exposure.
- Connections barely matters once the rest is in. The connections × arterial weight is a bit negative,
  which matches the arterial drop in notebook 05.
- Sidewalks: one side looked risky in notebook 05, but once road type is in, it's slightly negative.
  Those roads were mostly arterials.
- The features overlap, so single weights can move around. Look at direction and size together, not
  exact values.

## 8. Save

The split gets saved so Model B (road + image features) uses the exact same train and test areas.

In [ ]:
df[["segment_id", "block", "split"]].to_csv(PROCESSED / "segment_split.csv", index=False)

test_out = test[["segment_id", "road_type", "had_crash"]].assign(baseline=baseline_guess.values, model_a=model_guess)
test_out.to_csv(PROCESSED / "model_a_test_predictions.csv", index=False)
results.to_csv(PROCESSED / "model_a_results.csv", index=False)

## Next

- Try speed limit and lanes one at a time and keep them only if test PR-AUC goes up. (Decided on the
  training area, not the test area, so we don't tune to the test set.)
- Model B: same model, same split, plus the image features. The project's answer is Model B's PR-AUC
  minus Model A's.